<a href="https://colab.research.google.com/github/williansdb/ai_llm_sbom_provo/blob/main/ai_llm_sbom_provo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ AI & LLM Audit Framework (SBOM + PROV-O)
> **Arquivo:** `ai_llm_sbom_provo.ipynb` | **Autor:** Willian

---

### 📜 Termos de Uso e Atribuição
Este projeto é de código aberto para fins acadêmicos e profissionais. Ao utilizar ou adaptar este código, cite a fonte original vinculada ao repositório:

🔗 **GitHub:** [https://github.com/williansdb/ai_llm_som_provo](https://github.com/williansdb/ai_llm_som_provo)

---
### 📖 Sobre o Projeto
* **Descrição:** Framework para auditoria de proveniência em modelos de linguagem, integrando inventário de software dinâmico e rastreabilidade de dados.
* **Padrões:** Implementação baseada em **CycloneDX** (SBOM) e **PROV-O** (W3C Provenance Ontology).
* **Saída:** Geração automática de `relatorio_executivo.md` com formatação profissional para auditoria.
---

In [ ]:
#-------------BLOCO 1
from datetime import datetime #Biblioteca para detectar horário
from zoneinfo import ZoneInfo #Biblioteca para detectar fuso horário
from google.colab import output # Biblioteca que detecta seu local
from termcolor import colored # Biblioteca de cores
from tqdm.notebook import tqdm # Biblioteca para mostrar barra de progresso
import re # Importado para a limpeza do nome (sanitização)

# --- DEFINIÇÃO DE VARIÁVEIS GLOBAIS (O "Cérebro" do Notebook) ---
try:
    fuso_str = output.eval_js('Intl.DateTimeFormat().resolvedOptions().timeZone') # Captura o fuso horário local para todos os blocos
except:
    fuso_str = "UTC"

# Sanitização
usuario_raw = input("Digite seu nome: ") # Se identifique (auditoria)
usuario = re.sub(r'[^a-zA-ZÀ-ÿ\s]', '', usuario_raw) # Para remover caracteres especiais que podem quebrar o grafo (mantém letras, acentos e espaços)
usuario = usuario.strip()[:25] # Limita nome do usuário para 25 caracteres
if not usuario: # se o usuário não se identificar - escrever lixo ou deixar vazio
    usuario = "Visitante_Curioso"
    print(f"ℹ️ {colored('Nota:', 'yellow')} Nome não detectado. Registrado como {usuario}.")

# Mensagem de Saudação
def obter_saudacao():
    try:
        # Tenta detectar o fuso-horário via JS (javascript)
        fuso_str = output.eval_js('Intl.DateTimeFormat().resolvedOptions().timeZone')
        agora = datetime.now(ZoneInfo(fuso_str))
        hora = agora.hour
        if hora < 12:
            return "Bom dia"
        elif hora < 18:
            return "Boa tarde"
        else:
            return "Boa noite"
    except: # Se não der certo, retorna com um simples Olá sem identificar a saudação conforme horário
        return "Olá"

# Mensagem de instalação com adição de barra de progresso geral
def mensagem_instalacao():
    try:
        # 1. Tenta capturar o fuso e a data atual
        fuso_str = output.eval_js('Intl.DateTimeFormat().resolvedOptions().timeZone')
        agora = datetime.now(ZoneInfo(fuso_str))

        # 2. Formatação brasileira (Data e Hora)
        data_str = agora.strftime("%d/%m/%Y")
        hora_str = agora.strftime("%H:%M")

        # 3. Retorno com a data explícita (substituindo o "hoje")
        return (f"\n✅ Olá, {colored(usuario, 'blue')}, a instalação foi concluída com sucesso em "
                f"{data_str} às {colored(hora_str, 'green', attrs=['bold'])}!")

    except:
        # Fallback caso o JS do navegador falhe (usamos UTC ou data simples)
        data_simples = datetime.now().strftime("%d/%m/%Y")
        return f"\n✅ Olá, {colored(usuario, 'blue')}, a instalação foi registrada em {data_simples}!"

# --- PROCESSO DE INSTALAÇÃO ---

print(f"{colored(obter_saudacao(), 'blue', attrs=['bold'])}, {usuario}!")
print("Estamos preparando o motor das bibliotecas...\n")

# instala as bibliotecas necessárias
libs = ["cyclonedx-python-lib", "prov", "pydot", "graphviz", "transformers", "accelerate", "sentencepiece"]
# cyclonedx-python-lib: Segurança e inventário (SBOM).
# prov: Rastreamento de proveniência de dados.
# pydot: Criação de gráficos e diagramas.
# graphviz: Visualização de estruturas de dados).
# transformers: Modelos de Inteligência Artificial (NLP).
# accelerate: Alta performance e uso de GPU na IA.
# sentencepiece: Processamento de texto para modelos de linguagem.

# Criando a barra de progresso visual
# A cor da barra é definida no parâmetro 'colour'
with tqdm(total=len(libs), desc="Progresso", colour='green', bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{percentage:.0f}%]') as pbar:
    for lib in libs:
        !pip install {lib} -q # O -q deixa o pip quieto para não quebrar a barra
        pbar.update(1) # incrementa a barra em 1 unidade a cada bibliotaca instalada

print(mensagem_instalacao())

In [ ]:
#-------------BLOCO 2: Setup do Motor de IA (Resiliente)
import warnings, logging, torch
from transformers import pipeline, AutoTokenizer

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

print(f"⚙️ {colored(usuario, 'blue')}, preparando o motor de IA...")

# @markdown Defina o modelo (Tente "gpt2" para testar a compatibilidade)
modelo_nome = "Qwen/Qwen2.5-0.5B-Instruct"
# Modelos testados:
# "gpt2", "Qwen/Qwen2.5-0.5B-Instruct", "unsloth/Llama-3.2-1B-Instruct"

# 1. Carregar o Tokenizer primeiro para garantir o preenchimento (padding)
# Modelos antigos como GPT-2 não têm um pad_token definido por padrão
tokenizer = AutoTokenizer.from_pretrained(modelo_nome)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Criar o pipeline passando o tokenizer configurado
pipe = pipeline(
    "text-generation",
    model=modelo_nome,
    tokenizer=tokenizer, # Forçamos o uso do tokenizer que ajustamos acima
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.float32 # Forçamos float32 para modelos legacy como GPT-2
)

# 3. Verificação de Segurança para o Bloco 3
# Se o modelo não tiver template de chat, avisamos o sistema
has_chat_template = hasattr(pipe.tokenizer, "chat_template") and pipe.tokenizer.chat_template is not None

print(f"✅ {colored('Motor pronto!', 'green')} {colored(usuario, 'blue')}, o modelo {modelo_nome} já está na memória.")
if not has_chat_template:
    print(f"⚠️ {colored('Nota:', 'yellow')} Este modelo não suporta modo Chat. O Bloco 3 usará texto puro.")

In [ ]:
#-------------BLOCO 3: Inferência e Registro de Tempo (Versão Global Compliance)
import time
from datetime import datetime, timezone

pergunta_padrao = "O que é um número primo?"
pergunta_usuario = input(f"👤 {colored(usuario, 'blue')}, digite sua pergunta (ou Enter para o padrão): ") or pergunta_padrao

# 1. Ajuste de Formato de Entrada
if hasattr(pipe.tokenizer, "chat_template") and pipe.tokenizer.chat_template is not None:
    conteudo_para_ia = [{"role": "user", "content": pergunta_usuario}]
    modo_chat = True
else:
    conteudo_para_ia = pergunta_usuario
    modo_chat = False

print(f"🧠 {colored(usuario, 'blue')}, a IA está redigindo a resposta...")

# 2. Execução e Cronometragem (Padrão Global UTC para Proveniência)
# Captura o início com timezone-aware datetime para conformidade PROV-O
inicio_execucao = datetime.now(timezone.utc)
t0 = time.time()

saida = pipe(
    conteudo_para_ia,
    max_new_tokens=600,
    do_sample=True,
    temperature=0.7,
    pad_token_id=pipe.tokenizer.eos_token_id
)

t1 = time.time()
# Captura o fim em UTC para garantir a integridade do rastro de auditoria
fim_execucao = datetime.now(timezone.utc)
tempo_total = round(t1 - t0, 2)

# 3. Extração e Limpeza da Resposta
try:
    if modo_chat:
        # Para modelos que suportam chat_template (como Llama ou Gemma)
        resposta_texto = saida[0]['generated_text'][-1]['content']
    else:
        # Para modelos base (como o GPT-2 usado no seu framework)
        texto_gerado = saida[0]['generated_text']
        resposta_texto = texto_gerado.replace(pergunta_usuario, "").strip()
except (KeyError, IndexError, TypeError):
    resposta_texto = str(saida)

# 4. Exibição Final com Rastro de Auditoria
print(f"\n✅ {colored('Resposta concluída!', 'green')} (Tempo: {tempo_total}s)")
print(f"🕒 Registro Global (UTC): {inicio_execucao.strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 40)
print(resposta_texto)
print("-" * 40)

In [ ]:
#-------------BLOCO 4: Inventário de Software (SBOM CycloneDX - Auto-Discovery)
import json
from datetime import datetime, timezone
import importlib.metadata

# Definição centralizada das dependências críticas do framework
# Se você instalar algo novo, basta adicionar o nome aqui
libs_auditadas = ["transformers", "torch", "termcolor", "pydot", "numpy"]

print(f"📦 {colored(usuario, 'blue')}, executando varredura dinâmica no ambiente...")

componentes = []

# O motor de busca: varre as libs existentes e extrai as versões reais
for lib in libs_auditadas:
    try:
        versao = importlib.metadata.version(lib)
        componentes.append({
            "type": "library",
            "name": lib,
            "version": versao,
            "hashes": [{"alg": "SHA-256", "content": "verified_at_runtime"}],
            "externalReferences": [{"url": f"https://pypi.org/project/{lib}/", "type": "website"}]
        })
    except importlib.metadata.PackageNotFoundError:
        # Se uma lib crítica não for achada, registramos como alerta (importante para TCC)
        print(f"⚠️ {colored('Aviso:', 'yellow')} Biblioteca {lib} não encontrada no ambiente.")
        continue

# 2. Estrutura CycloneDX (Sincronizada com o Tempo da Inferência)
sbom_data = {
    "bomFormat": "CycloneDX",
    "specVersion": "1.4",
    "metadata": {
        "timestamp": inicio_execucao.isoformat(), # Garante o Não-Repúdio
        "tools": {
            "components": [{"name": "Framework_Auditoria_Fatec", "version": "1.0"}]
        },
        "component": {
            "type": "machine-learning-model",
            "name": modelo_nome
        }
    },
    "components": componentes
}

# 3. Persistência Física
with open("bom_projeto_ia.json", "w") as f:
    json.dump(sbom_data, f, indent=4)

print(f"✅ {colored('SBOM Sincronizado!', 'green')} {len(componentes)} componentes registrados com sucesso.")

In [ ]:
#------------- BLOCO 5: INTEGRAÇÃO: SBOM, PROV-O e Gráfico de Auditoria
import datetime
import json
import uuid
import pydot
from IPython.display import Image

# 1. Configurações Iniciais e Metadados
id_seguro = usuario.replace(" ", "_")
data_grafico_utc = inicio_execucao.strftime('%d/%m/%Y (UTC)')
arquivo_imagem = 'auditoria_final.png'

# 2. Estrutura PROV-O (Proveniência W3C)
prov_data = {
    "agent": {
        f"ex:{id_seguro}": { "prov:type": "prov:Person", "name": usuario },
        "ex:AI_Model": { "prov:type": "prov:SoftwareAgent", "name": modelo_nome }
    },
    "activity": {
        "ex:Inference_Task": {
            "prov:startTime": inicio_execucao.isoformat(),
            "prov:endTime": datetime.datetime.now().isoformat()
        }
    },
    "entity": {
        "ex:AI_Response": {
            "prov:wasGeneratedBy": "ex:Inference_Task",
            "prov:wasAssociatedWith": f"ex:{id_seguro}"
        }
    }
}

# 3. Geração do Gráfico Visual (pydot)
titulo_grafico = f"SISTEMA DE AUDITORIA E PROVENIÊNCIA DE IA\nAuditor: {usuario} | {data_grafico_utc}"
grafo = pydot.Dot(graph_type='digraph', label=titulo_grafico, labelloc='t', fontsize='14', rankdir="LR")

# Nós do Gráfico
n_user = pydot.Node(id_seguro, label=f"Auditor: {usuario}", shape='polygon', sides='5', fillcolor='gold', style='filled')
n_model = pydot.Node("Mod", label=f"Modelo:\n{modelo_nome}", shape='cylinder', fillcolor='lightyellow', style='filled')
n_act = pydot.Node("Acao", label=f"Inferência IA\n(PROV-O Activity)", shape='box', fillcolor='lightblue', style='filled')

grafo.add_node(n_user); grafo.add_node(n_model); grafo.add_node(n_act)
grafo.add_edge(pydot.Edge(n_act, n_user, label="wasAssociatedWith", color='blue'))
grafo.add_edge(pydot.Edge(n_act, n_model, label="usedEntity", color='red'))

# Cluster SBOM (CycloneDX) no Gráfico
sub_sbom = pydot.Cluster('sbom', label='Composição do Software (SBOM)', color='darkgreen', style='dashed')
for comp in componentes: # Usa a lista de componentes detectada no Bloco 4
    n_lib = pydot.Node(comp['name'], label=f"{comp['name']}\nv{comp['version']}", shape='component', fillcolor='#f0f0f0', style='filled')
    sub_sbom.add_node(n_lib)
    grafo.add_edge(pydot.Edge(n_model, n_lib, style='dotted', color='grey'))
grafo.add_subgraph(sub_sbom)

# 4. Salvamento do Relatório Final (JSON para o GitHub)
relatorio_final = {
    "bomFormat": "CycloneDX",
    "specVersion": "1.5",
    "metadata": { "timestamp": datetime.datetime.now().isoformat(), "authors": [{"name": usuario}] },
    "components": componentes,
    "provenance": prov_data
}

try:
    # Salva JSON e Imagem
    with open("relatorio_auditoria.json", "w") as f:
        json.dump(relatorio_final, f, indent=4)

    grafo.write_png(arquivo_imagem)
    display(Image(arquivo_imagem)) # Exibe o gráfico na tela!

    print(f"✅ {usuario}, Gráfico, JSON e SBOM gerados com sucesso!")

except Exception as e:
    print(f"❌ Erro na geração final: {e}")

In [ ]:
#-------------BLOCO 6: Relatório Executivo e Persistência (Versão Final Alinhada)
import json
import os
from datetime import datetime, timezone
from termcolor import colored

# 1. Definição dos Nomes de Arquivos Técnicos
timestamp_file = inicio_execucao.strftime('%Y%m%d_%H%M%S')
nome_arquivo_json = f"audit_log_{timestamp_file}.json"
nome_arquivo_md = "relatorio_executivo.md"

# 2. Lógica de Resiliência para Status e Horários
try:
    termino_str = fim_execucao.strftime('%H:%M:%S') + " UTC"
    status_auditoria = "SUCCESS"
except NameError:
    termino_str = "ABORTED_BY_POLICY"
    status_auditoria = "SECURITY_INCIDENT"

# 3. Preparação dos Dados para Transcrição (PROV-O e SBOM)
agentes = relatorio_final['provenance']['agent']
ativ = relatorio_final['provenance']['activity']['ex:Inference_Task']
lista_componentes = relatorio_final['components']

# 4. Construção do Conteúdo Markdown (Documento para o GitHub)
col1, col2, col3 = 20, 18, 22
conteudo_md = f"""# Relatório Executivo de Auditoria de IA
**Framework de Proveniência e Auditoria Técnica (PROV-O + CycloneDX)**

## 👤 Identificação do Agente
- **Auditor Responsável:** {usuario} (prov:Person)
- **Modelo Processador:** `{modelo_nome}` (prov:SoftwareAgent)
- **Data da Sessão:** {inicio_execucao.strftime('%d/%m/%Y')}

## ⚡ Performance e Integridade
- **Status:** `{status_auditoria}`
- **Tempo de Resposta:** {tempo_total}s
- **Início (UTC):** {inicio_execucao.strftime('%H:%M:%S')}
- **Término (UTC):** {termino_str}

## 🔍 Detalhes da Interação
### Entrada (Prompt):
> {pergunta_usuario if 'pergunta_usuario' in globals() else 'N/A'}

## 📦 Inventário de Software (SBOM)
| {"Componente".ljust(col1)} | {"Versão".ljust(col2)} | {"Status".ljust(col3)} |
| :{"-"*(col1-1)} | :{"-"*(col2-1)} | :{"-"*(col3-1)} |
"""

for comp in lista_componentes:
    conteudo_md += f"| {comp['name'].ljust(col1)} | {comp['version'].ljust(col2)} | {'Verificado'.ljust(col3)} |\n"

conteudo_md += f"\n---\n*Relatório gerado automaticamente para fins de auditoria e conformidade técnica.*"

# 5. Escrita dos Arquivos Físicos
with open(nome_arquivo_json, "w") as f:
    json.dump(relatorio_final, f, indent=4)

with open(nome_arquivo_md, "w", encoding="utf-8") as f:
    f.write(conteudo_md)

# 6. EXIBIÇÃO FORMATADA NO CONSOLE (Transcrição da Auditoria)
print(f"\n{'='*65}")
print(f"📄 RELATÓRIO DE AUDITORIA DE IA - {status_auditoria}")
print(f"📁 ARQUIVO TÉCNICO: {nome_arquivo_json}")
print(f"{'='*65}")

print(f"\n[ 👤 IDENTIFICAÇÃO E PROVENIÊNCIA ]")
print(f"• Auditor (Humano): {agentes[f'ex:{id_seguro}']['name']} [prov:Person]")
print(f"• Processador (IA): {agentes['ex:AI_Model']['name']} [prov:SoftwareAgent]")

print(f"\n[ ⚡ PERFORMANCE E INTEGRIDADE ]")
print(f"• Início:  {inicio_execucao.strftime('%H:%M:%S')} UTC")
print(f"• Término: {termino_str}")
print(f"• Tempo Total: {tempo_total}s")

print(f"\n[ 📦 INVENTÁRIO DE SOFTWARE (SBOM) ]")
for comp in lista_componentes:
    print(f"• {comp['name']}: v{comp['version']}")

print(f"\n{'='*65}")
print(f"✅ {colored('Log de Auditoria sincronizado e salvo com sucesso!', 'green')}")
print(f"📝 {colored('O arquivo ' + nome_arquivo_md + ' foi gerado com sucesso.', 'cyan')}")

In [ ]:
# @title 📦 Exportação de Evidências (Clique para expandir) { display-mode: "form" }
ativar_download = False # @param {type:"boolean"}

if ativar_download:
    import zipfile
    import os
    from google.colab import files
    from datetime import datetime, timezone
    from termcolor import colored

    # 1. Definição do Timestamp para o arquivo ZIP
    ts = inicio_execucao.strftime('%Y%m%d_%H%M%S') if 'inicio_execucao' in globals() else datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_manual')
    zip_nome = f"evidencias_auditoria_{ts}.zip"

    # 2. Lista de Alvos Sincronizada (Priorizando o Relatório Executivo)
    arquivos_alvo = [
        "relatorio_executivo.md",        # Seu relatório Markdown polido
        "auditoria_visual_final_pt.png", # Gráfico de Proveniência (Bloco 5)
        "bom_projeto_ia.json"            # SBOM Técnico (Bloco 4)
    ]

    # Adiciona o JSON de log se ele existir na sessão
    if 'nome_arquivo_json' in globals():
        arquivos_alvo.append(nome_arquivo_json)

    print(f"📦 {colored(usuario, 'blue')}, consolidando pacote de evidências...")

    try:
        with zipfile.ZipFile(zip_nome, 'w') as zipf:
            for arquivo in arquivos_alvo:
                if os.path.exists(arquivo):
                    zipf.write(arquivo)
                    print(f"  • {arquivo} {colored('anexado', 'green')}.")
                else:
                    # Alerta caso o arquivo principal não tenha sido gerado
                    print(f"  • {arquivo} {colored('não localizado', 'red')}.")

        print(f"\n✅ {colored('Sucesso!', 'green')} O download do pacote de auditoria iniciará.")
        files.download(zip_nome)

    except Exception as e:
        print(f"❌ Falha na exportação: {e}")
else:
    print(f"💤 {colored('Bloco de Exportação Inativo.', 'yellow')} (Marque o checkbox para baixar as evidências).")